# gpt_oss CoT-collapse compression probe (CPU, 0 GPU quota)

Runs `tools/probe_gptoss_compression.py`: replays our shipped collapse template vs plain (no collapse) vs several MORE-aggressive forged-Harmony variants on the real gpt_oss GGUF at hop-0, and asks whether any variant generates fewer tokens (cheaper 8-hop replay -> more findings/model) while STILL firing the exact-payload http.post. A win must both stay at fire-rate ~1.0 AND beat `current` on warm per-candidate cost.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
def find_repo_root() -> Path:
    for c in [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/AI-Agent-Security')]:
        if (c / 'tools' / 'probe_gptoss_compression.py').exists():
            return c
    raise FileNotFoundError('tools/probe_gptoss_compression.py not found; run the bootstrap cell first')
ROOT = find_repo_root(); os.chdir(ROOT); print('repo root:', ROOT)
print('gpu:', 'none (CPU kernel)' if not shutil.which('nvidia-smi') else subprocess.run(['nvidia-smi','-L'],text=True,capture_output=True).stdout.strip())
print('cpu count:', os.cpu_count())


In [ ]:
os.environ.setdefault('GPT_OSS_GGUF_REPO', 'unsloth/gpt-oss-20b-GGUF')
os.environ.setdefault('GPT_OSS_GGUF_FILE', 'gpt-oss-20b-Q4_K_M.gguf')
os.environ.setdefault('LLAMA_CPP_EXTRA_INDEX_URL', 'https://abetlen.github.io/llama-cpp-python/whl/cpu')


In [ ]:
import importlib.util
def ensure_llama_cpp():
    if importlib.util.find_spec('llama_cpp') is not None:
        print('llama_cpp already installed'); return
    idx = os.getenv('LLAMA_CPP_EXTRA_INDEX_URL','https://abetlen.github.io/llama-cpp-python/whl/cpu')
    print('installing llama-cpp-python (CPU) from', idx)
    try:
        subprocess.run([sys.executable,'-m','pip','install','-q','--prefer-binary','llama-cpp-python','--extra-index-url',idx], check=True)
    except subprocess.CalledProcessError:
        env=os.environ.copy(); env['CMAKE_ARGS']='-DGGML_CUDA=off'; env['FORCE_CMAKE']='1'
        subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--force-reinstall','llama-cpp-python'], check=True, env=env)
    if importlib.util.find_spec('llama_cpp') is None: raise ModuleNotFoundError('llama_cpp')
ensure_llama_cpp()


In [ ]:
import json
cmd = [sys.executable, 'tools/probe_gptoss_compression.py',
       '--n','10','--model','gpt_oss','--budget-per-model','3000',
       '--max-tool-hops','1','--min-fire-rate','0.99',
       '--out','research/results/gptoss-compression.latest.json',
       '--raw-out','research/results/gptoss-compression.raw.jsonl']
print('running:', ' '.join(cmd))
proc = subprocess.run(cmd, text=True)  # exit 0 = a variant beats current, 2 = tapped
print('probe exit code:', proc.returncode, '(0 = a faster-still-firing variant found, 2 = current is already minimal)')


In [ ]:
import shutil
p = Path('research/results/gptoss-compression.latest.json')
s = json.loads(p.read_text()); print(json.dumps(s, indent=2, sort_keys=True))
print('\n=== VERDICT ===')
print('a_variant_beats_current:', s['ranking']['a_variant_beats_current'])
for row in s['ranking']['ranked_qualifying']:
    print(f"  {row['template']:16} fire={row['hit_rate']} warm={row['warm_cost_s']}s speedup_vs_current={row['speedup_vs_current']}")
print('  disqualified (below fire floor):', s['ranking']['disqualified'])
out = Path('/kaggle/working')
if out.exists():
    for f in [p, Path('research/results/gptoss-compression.raw.jsonl')]:
        if f.exists() and f.resolve()!=(out/f.name).resolve(): shutil.copy(f, out/f.name)
    print('copied outputs to', out)
